This is a starter notebook for the project, you'll have to import the libraries you'll need, you can find a list of the ones available in this workspace in the requirements.txt file in this workspace. 

In [119]:
import os
OPENAI_API_KEY="voc-17929747421266773738297677bfe6acc1cf4.88609518"
os.environ["OPENAI_API_KEY"] = "voc-17929747421266773738297677bfe6acc1cf4.88609518"
os.environ["OPENAI_API_BASE"] = "https://openai.vocareum.com/v1"

from langchain.llms import OpenAI


In [195]:
from pydantic import BaseModel, Field,NonNegativeInt
from typing import List


In [188]:
#print(schema)

class RealEstateListing(BaseModel):
    """
    A real estate listing.
    
    Attributes:
    - neighborhood: str
    - price: NonNegativeInt
    - bedrooms: NonNegativeInt
    - bathrooms: NonNegativeInt
    - house_size: NonNegativeInt
    - description: str
    - neighborhood_description: str
    """
    neighborhood: str = Field(description="The neighborhood where the property is located")
    price: NonNegativeInt = Field(description="The price of the property in USD")
    bedrooms: NonNegativeInt = Field(description="The number of bedrooms in the property")
    bathrooms: NonNegativeInt = Field(description="The number of bathrooms in the property")
    house_size: NonNegativeInt = Field(description="The size of the house in square feet")
    description: str = Field(description="A description of the property")
    neighborhood_description: str = Field(description="A description of the neighborhood.")  

class ListingCollection(BaseModel):
    """
    A collection of real estate listings.
    
    Attributes:
    - listings: List[RealEstateListing]
    """
    listings: List[RealEstateListing] = Field(description="A list of real estate listings")

In [189]:

from langchain.prompts import PromptTemplate
llm=OpenAI(model_name='gpt-3.5-turbo',api_key=OPENAI_API_KEY) 

prompt='''I want to generate atleast {no} csv examples as below  , please dont add anything extra to the data and make sure all the data should be same format
Keep one additional spaces between examples 
example 

Neighborhood: Green Oaks,Price: $800,000,Bedrooms: 3,Bathrooms: 2,House Size: 2,000 sqft,Description: Welcome to this eco-friendly oasis nestled in the heart of Green Oaks. This charming 3-bedroom, 2-bathroom home boasts energy-efficient features such as solar panels and a well-insulated structure. Natural light floods the living spaces, highlighting the beautiful hardwood floors and eco-conscious finishes. The open-concept kitchen and dining area lead to a spacious backyard with a vegetable garden, perfect for the eco-conscious family. Embrace sustainable living without compromising on style in this Green Oaks gem.,Neighborhood Description: Green Oaks is a close-knit, environmentally-conscious community with access to organic grocery stores, community gardens, and bike paths. Take a stroll through the nearby Green Oaks Park or grab a cup of coffee at the cozy Green Bean Cafe. With easy access to public transportation and bike lanes, commuting is a breeze.

'''

from langchain.output_parsers import PydanticOutputParser
pydantic_parser = PydanticOutputParser(pydantic_object=ListingCollection)

# Get the JSON schema from the Pydantic BaseModel
#schema = CustomerSupportTicket.schema_json(indent=2)

# Define the prompt template
synthetic_data_template = PromptTemplate(
    template='{prompt}\n{format_instructions}',
    input_variables=["prompt"],
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()}
)

_input = synthetic_data_template.format_prompt(prompt=prompt.format(no=13))

In [176]:
#print(_input.text)

In [190]:
data=llm(_input.text)

In [194]:
from fastapi.encoders import jsonable_encoder
result = pydantic_parser.parse(data)
df = pd.DataFrame(jsonable_encoder(result.listings))
df.head()

,neighborhood,price,bedrooms,bathrooms,house_size,description,neighborhood_description
0,Green Oaks,800000,3,2,2000,Welcome to this eco-friendly oasis nestled in ...,"Green Oaks is a close-knit, environmentally-co..."
1,Sunnyvale,900000,4,3,2500,Beautiful family home in the heart of Sunnyval...,"Sunnyvale is known for its excellent schools, ..."
2,Rockridge,1200000,5,4,3000,"Luxurious 5-bedroom, 4-bathroom home in the pr...",Rockridge is a highly sought-after neighborhoo...
3,Laurel Heights,1500000,6,5,3500,"Stunning 6-bedroom, 5-bathroom home in the exc...",Laurel Heights is a prestigious neighborhood k...
4,Nob Hill,1800000,4,3,2800,"Impeccably designed 4-bedroom, 3-bathroom home...",Nob Hill is a historic neighborhood known for ...


In [196]:
df

,neighborhood,price,bedrooms,bathrooms,house_size,description,neighborhood_description
0,Green Oaks,800000,3,2,2000,Welcome to this eco-friendly oasis nestled in ...,"Green Oaks is a close-knit, environmentally-co..."
1,Sunnyvale,900000,4,3,2500,Beautiful family home in the heart of Sunnyval...,"Sunnyvale is known for its excellent schools, ..."
2,Rockridge,1200000,5,4,3000,"Luxurious 5-bedroom, 4-bathroom home in the pr...",Rockridge is a highly sought-after neighborhoo...
3,Laurel Heights,1500000,6,5,3500,"Stunning 6-bedroom, 5-bathroom home in the exc...",Laurel Heights is a prestigious neighborhood k...
4,Nob Hill,1800000,4,3,2800,"Impeccably designed 4-bedroom, 3-bathroom home...",Nob Hill is a historic neighborhood known for ...
5,North Beach,950000,2,1,1500,"Charming 2-bedroom, 1-bathroom bungalow in the...",North Beach is a lively neighborhood known for...
6,Pacific Heights,2000000,4,2,2200,"Elegant 4-bedroom, 2-bathroom home in the pres...",Pacific Heights is an affluent neighborhood kn...
7,Russian Hill,1600000,3,3,2400,"Modern 3-bedroom, 3-bathroom townhouse in the ...",Russian Hill is a trendy neighborhood known fo...
8,Sunset District,1000000,4,2,2000,"Spacious 4-bedroom, 2-bathroom home in the fam...",Sunset District is a diverse neighborhood know...
9,Tenderloin,750000,2,1,1200,"Cozy 2-bedroom, 1-bathroom condo in the vibran...",Tenderloin is a diverse neighborhood known for...


In [197]:
Parse_data=df

In [198]:
Parse_data.to_csv("Sample_data.csv")

In [201]:
df

,neighborhood,price,bedrooms,bathrooms,house_size,description,neighborhood_description
0,Green Oaks,800000,3,2,2000,Welcome to this eco-friendly oasis nestled in ...,"Green Oaks is a close-knit, environmentally-co..."
1,Sunnyvale,900000,4,3,2500,Beautiful family home in the heart of Sunnyval...,"Sunnyvale is known for its excellent schools, ..."
2,Rockridge,1200000,5,4,3000,"Luxurious 5-bedroom, 4-bathroom home in the pr...",Rockridge is a highly sought-after neighborhoo...
3,Laurel Heights,1500000,6,5,3500,"Stunning 6-bedroom, 5-bathroom home in the exc...",Laurel Heights is a prestigious neighborhood k...
4,Nob Hill,1800000,4,3,2800,"Impeccably designed 4-bedroom, 3-bathroom home...",Nob Hill is a historic neighborhood known for ...
5,North Beach,950000,2,1,1500,"Charming 2-bedroom, 1-bathroom bungalow in the...",North Beach is a lively neighborhood known for...
6,Pacific Heights,2000000,4,2,2200,"Elegant 4-bedroom, 2-bathroom home in the pres...",Pacific Heights is an affluent neighborhood kn...
7,Russian Hill,1600000,3,3,2400,"Modern 3-bedroom, 3-bathroom townhouse in the ...",Russian Hill is a trendy neighborhood known fo...
8,Sunset District,1000000,4,2,2000,"Spacious 4-bedroom, 2-bathroom home in the fam...",Sunset District is a diverse neighborhood know...
9,Tenderloin,750000,2,1,1200,"Cozy 2-bedroom, 1-bathroom condo in the vibran...",Tenderloin is a diverse neighborhood known for...


### appending this data to Chroma DB 

In [199]:
from langchain.docstore.document import Document
from langchain.llms import OpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.evaluation import load_evaluator
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain.vectorstores.chroma import Chroma

In [203]:
doc_list=[Document(page_content =Parse_data["description"][i], metadata={'id': str(i)}) for i in range(0,len(Parse_data["description"]))]
#doc_list=[]
splitter=CharacterTextSplitter(chunk_size=100,chunk_overlap=0)
splits_docs=splitter.split_documents(doc_list)

In [204]:
embedding=OpenAIEmbeddings()
db=Chroma.from_documents(splits_docs,embedding)

In [205]:
query= " I am looking for 2 Bedrooms appartment in Good society"

In [206]:
prompt= '''Think as real state agent you need to answer below 
{query}

context: {context}
'''

In [207]:
def get_response(query,db,prompt):
    
    result=db.similarity_search(query,k=2)
    sources = [i.metadata['id'] for i in result if len(i.metadata) > 0]
    context="\n\n---\n\n".join([doc.page_content for doc in result])
    new_prompt=prompt.format(query=query,context=context)
    
    print(llm(new_prompt)+"\n" + "sources:{sources}".format(sources=sources))
    
    

In [208]:
 get_response(query,db,prompt)

As a real estate agent, I would recommend the sleek and modern 2-bedroom condo in the Yerba Buena neighborhood for someone looking for a modern and luxurious apartment. The amenities such as the rooftop terrace, fitness center, and concierge services make it a great option for someone looking for a high-end living experience.

Alternatively, for someone looking for a charming and cozy home in a vibrant neighborhood, the 2-bedroom bungalow in North Beach would be a great fit. The sunny living room, remodeled kitchen, and private backyard with a deck offer a warm and inviting living space. The proximity to cafes, shops, and parks adds to the convenience and charm of this property.
sources:['12', '5']


### Augmented Prompt template  

In [209]:
AUGMENT_PROMPT_TEMPLATE =\
"""
Based on the following context:

{context}

---

craft a response that not only answers the question {question}, but also ensures that your explanation is distinct, captivating, and customized to align with the specified preferences. This involves subtly emphasizing aspects of the property that align with what the buyer is looking for.
"""

In [210]:
Augmented=AUGMENT_PROMPT_TEMPLATE.format(context=context,question=query)

In [211]:
llm(Augmented)

"I am thrilled to introduce you to the perfect match for your search - a stunning 2-bedroom loft nestled in the heart of a vibrant downtown neighborhood. This exquisite space boasts exposed brick walls, high ceilings, and large windows that offer breathtaking city views, creating a truly captivating urban living experience.\n\nThe seamless open floor plan effortlessly connects the living, dining, and kitchen areas, providing a welcoming atmosphere that is ideal for entertaining guests. With easy access to trendy restaurants, shopping destinations, and cultural attractions just a short walk away, you'll find yourself immersed in the dynamic energy of city life while still enjoying the comfort of a cozy and inviting home.\n\nThis stylish loft not only meets your criteria for a 2-bedroom apartment but also exceeds expectations with its unique charm and convenient location in a well-established and desirable neighborhood. Prepare to be impressed and make this urban oasis your own sanctuary